# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fraz-Rasool/ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Chosen lane: CTR / Engagement Opportunity Scoring**

I frame this as a **scoring problem**. The output would be a continuous opportunity score for each content page, where a higher score means the page is more worth reviewing for an engagement improvement.

The score is intended to combine signals such as search position, impressions, clicks, CTR, sessions, engagement rate, and other page-level context. The goal is not to predict Google's ranking or claim that a page will definitely improve. The goal is to **prioritize a finite review queue**.

This is different from a simple classification such as "good/bad page" because the action is naturally ordered: if a team can review 20 pages, the most useful output is a ranked/scored shortlist rather than a yes/no label for every page.

In [1]:
# Colab setup + starter data
# The repository is cloned into the Colab runtime, so paths are independent of
# where the notebook was opened from in GitHub.

from pathlib import Path
import subprocess
import pandas as pd
import numpy as np

REPO_ROOT = Path("/content/ML-Internship")

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/Fraz-Rasool/ML-Internship.git", str(REPO_ROOT)],
        check=True,
    )

DATA_PATH = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Starter CSV not found at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nRequired lane fields present:")
required = ["content_id", "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier"]
print(df[required].notna().all().to_string())


Rows: 30,000
Columns: 44

Required lane fields present:
content_id         True
impressions_90d    True
clicks_90d         True
ctr                True
avg_position       True
position_tier      True


## 2. Target or proxy

The eventual target should be an **observed post-action engagement outcome** measured after a page is reviewed and changed, ideally using a time-separated evaluation window.

The starter dataset does not contain a clean before/after intervention label, so I should **not pretend that it does**. For this stage, I use a defined proxy: a **position-adjusted CTR opportunity gap**. The proxy compares a page's observed CTR with a simple expected CTR for its position tier. A larger positive gap indicates that the page is performing below the level suggested by comparable position context and may deserve investigation.

This is only a screening proxy. It is not a causal label and does not prove that changing the page will increase CTR. In later work, I would prefer an outcome measured on a future time window or a documented intervention/review outcome.

In [2]:
# Sketch the provisional target/proxy.
# CTR is stored on a 0-100 percentage scale in the starter data.

tier_baseline = (
    df.groupby("position_tier", dropna=False)
      .apply(
          lambda g: pd.Series({
              "tier_impressions": g["impressions_90d"].sum(),
              "tier_clicks": g["clicks_90d"].sum(),
          }),
          include_groups=False,
      )
      .reset_index()
)

tier_baseline["expected_ctr_pct"] = (
    tier_baseline["tier_clicks"] /
    tier_baseline["tier_impressions"] * 100
)

target_sketch = df[
    [
        "content_id",
        "position_tier",
        "avg_position",
        "impressions_90d",
        "clicks_90d",
        "ctr",
    ]
].copy()

target_sketch = target_sketch.merge(
    tier_baseline[["position_tier", "expected_ctr_pct"]],
    on="position_tier",
    how="left",
)

target_sketch["ctr_opportunity_gap_pct"] = (
    target_sketch["expected_ctr_pct"] - target_sketch["ctr"]
)

target_sketch["high_opportunity_proxy"] = (
    (target_sketch["impressions_90d"] >= 500)
    & (target_sketch["avg_position"] > 0)
    & (target_sketch["avg_position"] <= 20)
    & (target_sketch["ctr_opportunity_gap_pct"] > 0)
)

print("Provisional target/proxy sketch:")
display(target_sketch.head(10))

print(
    f"Pages flagged high-opportunity by the provisional proxy: "
    f"{target_sketch['high_opportunity_proxy'].sum():,} "
    f"({target_sketch['high_opportunity_proxy'].mean() * 100:.2f}%)"
)


Provisional target/proxy sketch:


,content_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr,expected_ctr_pct,ctr_opportunity_gap_pct,high_opportunity_proxy
0,content_304f48230142,striking,10.6,3803,29,0.76,0.346876,-0.413124,False
1,content_a1fb4e703a9e,page_3_5,20.3,15320,7,0.05,0.154905,0.104905,False
2,content_9aa793d4d895,page_3_5,36.5,12581,11,0.09,0.154905,0.064905,False
3,content_331d6c4de07b,page_1,6.2,11751,58,0.49,0.350324,-0.139676,False
4,content_d99b7a2d90ca,page_3_5,44.0,19140,24,0.13,0.154905,0.024905,False
5,content_d4084a4bc775,page_1,8.5,3970,1,0.03,0.350324,0.320324,True
6,content_9a34b442b552,page_1,7.0,20,0,0.00,0.350324,0.350324,False
7,content_a63219c6e95a,page_3_5,21.2,1724,1,0.06,0.154905,0.094905,False
8,content_5e6c160719bc,page_3_5,46.0,32574,29,0.09,0.154905,0.064905,False
9,content_c27558df2b0c,page_1,4.9,1240,2,0.16,0.350324,0.190324,True


Pages flagged high-opportunity by the provisional proxy: 8,409 (28.03%)


## 3. Success metric

My primary provisional success metric is **Precision@K**.

Here, K represents the number of pages the content team can realistically review in a batch. Precision@K asks: **of the top K pages produced by the scoring system, what fraction are actually high-opportunity pages under the evaluation target/proxy?**

This matches the decision because review capacity is limited. A useful system should make the top of the queue trustworthy rather than merely producing a good average prediction across every page.

For the final model, K should be fixed before evaluation and chosen to match the available review capacity. The starter-data proxy will be treated as provisional; future-window outcomes are preferable for the final evaluation.

In [3]:
# Show the decision metric definition on the starter data.
# This is a framing check, not a claim that the proxy is the final evaluation label.

K = min(100, len(target_sketch))

print("Primary metric: Precision@K")
print(f"Illustrative review capacity K: {K}")
print(
    "Interpretation: among the top K pages selected by a future scoring model, "
    "the fraction that are genuinely high-opportunity under the agreed evaluation target."
)

# A simple transparent baseline for comparison:
baseline_mask = (
    (target_sketch["impressions_90d"] >= 500)
    & (target_sketch["avg_position"] > 0)
    & (target_sketch["avg_position"] <= 20)
    & (target_sketch["ctr"] < 0.5)
)

print(f"\nTransparent baseline rule candidates: {baseline_mask.sum():,}")
print("The final model should be compared against this kind of simple baseline.")


Primary metric: Precision@K
Illustrative review capacity K: 100
Interpretation: among the top K pages selected by a future scoring model, the fraction that are genuinely high-opportunity under the agreed evaluation target.

Transparent baseline rule candidates: 9,759
The final model should be compared against this kind of simple baseline.


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one row = one content page/content item (`content_id`).**

Each row represents the measurable performance context of one page over the starter dataset's observation window. The scoring output would therefore be attached to a page and used to decide whether that page should enter the review queue.

The code below loads the lane's starter-data slice and creates a small target/proxy sketch. I keep identifiers and measured features visible so it is clear what the model would actually score.

In [4]:
# Real dataframe showing the unit of analysis:
# one row = one content page/content item.

lane_slice = df[
    [
        "content_id",
        "position_tier",
        "avg_position",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "sessions_90d",
        "engagement_rate",
    ]
].copy()

print("Unit of analysis: ONE ROW = ONE CONTENT PAGE / CONTENT ITEM")
print(f"Rows in lane slice: {len(lane_slice):,}")

display(lane_slice.head(10))


Unit of analysis: ONE ROW = ONE CONTENT PAGE / CONTENT ITEM
Rows in lane slice: 30,000


,content_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr,sessions_90d,engagement_rate
0,content_304f48230142,striking,10.6,3803,29,0.76,17,5.88
1,content_a1fb4e703a9e,page_3_5,20.3,15320,7,0.05,9,0.00
2,content_9aa793d4d895,page_3_5,36.5,12581,11,0.09,11,0.00
3,content_331d6c4de07b,page_1,6.2,11751,58,0.49,78,1.28
4,content_d99b7a2d90ca,page_3_5,44.0,19140,24,0.13,145,0.00
5,content_d4084a4bc775,page_1,8.5,3970,1,0.03,5,0.00
6,content_9a34b442b552,page_1,7.0,20,0,0.00,1,0.00
7,content_a63219c6e95a,page_3_5,21.2,1724,1,0.06,28,3.57
8,content_5e6c160719bc,page_3_5,46.0,32574,29,0.09,68,5.88
9,content_c27558df2b0c,page_1,4.9,1240,2,0.16,3,0.00


## 5. Why ML beats a fixed rule here

A fixed rule such as **"position <= 20 and CTR < 0.5%"** is useful as a transparent screening baseline, but it treats very different pages as equivalent.

For example, two pages can have the same CTR but very different impressions, positions, sessions, and engagement context. A single threshold also creates abrupt boundaries: a page just above the threshold is treated as completely different from one just below it.

A scoring model can learn **interactions and gradual differences across multiple signals** and produce an ordered review queue. That is valuable when the team has limited review capacity.

However, ML is only justified if it beats a transparent baseline on an agreed evaluation metric. If the model does not improve the decision over a simple rule, the rule may be preferable because it is easier to explain and maintain.

In [5]:
# Compare the proposed ML framing with a transparent fixed-rule baseline.

rule_candidates = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

print(f"Fixed-rule candidates: {rule_candidates.sum():,}")
print(
    "The rule is intentionally kept as a baseline. "
    "ML is only worthwhile if a learned score improves the agreed decision metric "
    "on data that is separated appropriately from the training period."
)


Fixed-rule candidates: 9,759
The rule is intentionally kept as a baseline. ML is only worthwhile if a learned score improves the agreed decision metric on data that is separated appropriately from the training period.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.